# Thresholding, Morphology, Contours and Object Counting (OpenCV)

For each of the three pictures, separately, this notebook:

1. Converts to grayscale and applies **thresholding**
2. Applies **erosion, dilation, opening, closing**
3. Finds **contours** and **counts objects**

| Picture | What it is | What we count |
|---|---|---|
| 1 | Photo of leaves against the sky | dark leaf blobs |
| 2 | Photo of a printed sheet with dot clouds | individual dots, and dot clusters |
| 3 | Blank form template | black square/rectangle markers |

**Upload the three pictures in this order: leaves, dot-scatter sheet, blank form.**

## 0. Setup and helper functions

In [ ]:
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(imgs, titles, cols=3, w=5, h=4.5):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(w * cols, h * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        if im.ndim == 2:
            plt.imshow(im, cmap="gray")
        else:
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(t, fontsize=11)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

def morphology(binary, ksize, shape=cv2.MORPH_RECT, iterations=1):
    """Erosion, dilation, opening (erode->dilate), closing (dilate->erode)."""
    k = cv2.getStructuringElement(shape, (ksize, ksize))
    return {
        "erosion":  cv2.erode(binary, k, iterations=iterations),
        "dilation": cv2.dilate(binary, k, iterations=iterations),
        "opening":  cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k, iterations=iterations),
        "closing":  cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k, iterations=iterations),
    }

def count_objects(binary, original, min_area=20, max_area=None, number=True):
    """Find external contours, filter by area, draw them and return (count, contours, image)."""
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    keep = [c for c in cnts
            if cv2.contourArea(c) >= min_area and (max_area is None or cv2.contourArea(c) <= max_area)]
    out = original.copy()
    cv2.drawContours(out, keep, -1, (0, 0, 255), 2)
    if number:
        for i, c in enumerate(keep, 1):
            M = cv2.moments(c)
            if M["m00"]:
                cv2.putText(out, str(i), (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
    return len(keep), keep, out

results = {}  # summary of counts

## 1. Upload the pictures

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # select all 3 images (leaves, dot sheet, blank form)
    paths = list(uploaded.keys())
except ImportError:                    # running outside Colab
    paths = sorted(glob.glob("/mnt/user-data/uploads/*"))

assert len(paths) >= 3, "Please upload 3 images"
IMG_LEAVES, IMG_SCATTER, IMG_FORM = paths[:3]

img1 = cv2.imread(IMG_LEAVES)
img2 = cv2.imread(IMG_SCATTER)
img3 = cv2.imread(IMG_FORM)
show([img1, img2, img3], ["Picture 1 (leaves)", "Picture 2 (dot sheet)", "Picture 3 (blank form)"])

---
## Picture 1: Leaves

Dark (dried) leaves are darker than the sky and the yellow leaves, so a **global Otsu threshold** with inverse
binarisation turns them white. Opening removes tiny specks; closing fills small holes.

In [ ]:
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
blur1 = cv2.GaussianBlur(gray1, (5, 5), 0)

# Otsu picks the threshold automatically; set MANUAL_T1 to a number (0-255) to override
MANUAL_T1 = None
if MANUAL_T1 is None:
    T1, bin1 = cv2.threshold(blur1, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
else:
    T1, bin1 = cv2.threshold(blur1, MANUAL_T1, 255, cv2.THRESH_BINARY_INV)
print("Threshold used:", T1)

show([gray1, blur1, bin1], ["Grayscale", "Gaussian blur", f"Threshold (T={T1:.0f}, inverted)"])

In [ ]:
K1 = 3
m1 = morphology(bin1, K1)
show([bin1, m1["erosion"], m1["dilation"], m1["opening"], m1["closing"]],
     ["Binary", f"Erosion ({K1}x{K1})", f"Dilation ({K1}x{K1})", f"Opening ({K1}x{K1})", f"Closing ({K1}x{K1})"],
     cols=3)

In [ ]:
MIN_AREA1 = 30   # ignore blobs smaller than this many pixels

# Count on each version to see how morphology changes the result
for name, b in [("threshold only", bin1), ("erosion", m1["erosion"]), ("dilation", m1["dilation"]),
                ("opening", m1["opening"]), ("closing", m1["closing"])]:
    n, _, _ = count_objects(b, img1, MIN_AREA1)
    print(f"{name:15s}: {n} objects")

# Opening = cleanest result (removes specks without merging leaves)
n1, cnts1, out1 = count_objects(m1["opening"], img1, MIN_AREA1)
results["Picture 1: dark leaf blobs (after opening)"] = n1
show([img1, out1], ["Original", f"Contours after opening: {n1} objects"], cols=2, w=7, h=8)

---
## Picture 2: Dot sheet (photographed at an angle)

Lighting is uneven, so a **global threshold fails**; we use an **adaptive threshold**. The sheet also contains a frame,
rulers and corner squares that are *not* dots, so we first find the big frame contour and keep only the area inside it.
Then we count:

* **individual dots** (contours; overlapping dots merge, so this is a lower bound)
* **clusters** (smoothed dot-density map, one peak per cluster, then watershed to outline each cluster)

In [ ]:
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
blur2 = cv2.GaussianBlur(gray2, (3, 3), 0)

bin2_all = cv2.adaptiveThreshold(blur2, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV, blockSize=51, C=15)
show([gray2, bin2_all], ["Grayscale", "Adaptive threshold (everything dark)"], cols=2, w=7, h=5.5)

In [ ]:
def frame_mask(binary, shrink_frac=0.03):
    """Find the big black frame, return a mask of its INSIDE (frame line itself removed)."""
    h, w = binary.shape
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best, best_area = None, 0
    for c in cnts:
        a = cv2.contourArea(c)
        if 0.05 * h * w < a < 0.7 * h * w and a > best_area:
            best, best_area = c, a
    hull = cv2.convexHull(best)
    box = None
    for eps in np.linspace(0.01, 0.1, 30):          # simplify hull to a 4-corner polygon
        approx = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True)
        if len(approx) == 4:
            box = approx.reshape(4, 2).astype(np.int32)
            break
    if box is None:
        box = cv2.boxPoints(cv2.minAreaRect(best)).astype(np.int32)
    mask = np.zeros_like(binary)
    cv2.fillPoly(mask, [box], 255)
    k = int(shrink_frac * min(h, w)) * 2 + 1         # erosion strips the frame line
    mask = cv2.erode(mask, np.ones((k, k), np.uint8))
    return mask, box

mask2, box2 = frame_mask(bin2_all)
bin2 = cv2.bitwise_and(bin2_all, mask2)

vis = img2.copy()
cv2.polylines(vis, [box2], True, (0, 255, 0), 3)
show([vis, bin2], ["Detected frame", "Dots only (inside frame)"], cols=2, w=7, h=5.5)

In [ ]:
K2 = 3
m2 = morphology(bin2, K2, shape=cv2.MORPH_ELLIPSE)
show([bin2, m2["erosion"], m2["dilation"], m2["opening"], m2["closing"]],
     ["Binary", f"Erosion ({K2}x{K2})", f"Dilation ({K2}x{K2})", f"Opening ({K2}x{K2})", f"Closing ({K2}x{K2})"],
     cols=3, w=6, h=4.5)

In [ ]:
# --- Individual dots: contours on the binary image
MIN_AREA2 = 3
n_dots, cnts_dots, out_dots = count_objects(bin2, img2, MIN_AREA2, number=False)
results["Picture 2: individual dots (contours)"] = n_dots
print("Individual dots found:", n_dots, "(overlapping dots merge, so the true number is higher)")
show([img2, out_dots], ["Original", f"Dot contours: {n_dots}"], cols=2, w=8, h=6)

In [ ]:
# --- Clusters: dilate/blur dots into a density map, find one peak per cluster
SIGMA = 20        # smoothing (px); larger = coarser clusters
MIN_DIST = 60     # minimum distance between two cluster centres (px)
PEAK_FRAC = 0.4   # peak must be at least this fraction of the max density

dens = cv2.GaussianBlur(bin2.astype(np.float32), (0, 0), SIGMA)
local_max = cv2.dilate(dens, np.ones((MIN_DIST, MIN_DIST), np.uint8))
peaks = np.argwhere((dens == local_max) & (dens > PEAK_FRAC * dens.max()))   # (row, col)
n_clusters = len(peaks)

# Watershed: grow each peak into a region, then outline it with a contour
_, blob = cv2.threshold(cv2.GaussianBlur(bin2, (0, 0), 6), 0.2 * 255 * 0.5, 255, cv2.THRESH_BINARY)
markers = np.zeros(bin2.shape, np.int32)
for i, (r, c) in enumerate(peaks, 1):
    cv2.circle(markers, (int(c), int(r)), 5, i, -1)
ws = cv2.watershed(cv2.cvtColor(cv2.dilate(blob, np.ones((5, 5), np.uint8)), cv2.COLOR_GRAY2BGR), markers)

out_cl = img2.copy()
colors = [(0, 0, 255), (0, 200, 0), (255, 0, 0), (0, 165, 255), (255, 0, 255), (0, 255, 255)]
for i, (r, c) in enumerate(peaks, 1):
    region = ((ws == i) & (blob > 0)).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out_cl, cnts, -1, colors[(i - 1) % len(colors)], 3)
    cv2.putText(out_cl, f"#{i}", (int(c) - 15, int(r)), cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                colors[(i - 1) % len(colors)], 2, cv2.LINE_AA)

results["Picture 2: dot clusters"] = n_clusters
print("Clusters found:", n_clusters)
show([dens, out_cl], ["Dot density map (peaks = cluster centres)", f"Clusters: {n_clusters}"], cols=2, w=8, h=6)

---
## Picture 3: Blank form template

Black-on-white, so **Otsu** works perfectly. The form contains big corner squares, small rectangular markers,
thin frame lines, ruler ticks and text. **Opening with a 9x9 kernel** deletes everything thinner than 9 px
(lines, ticks, text) and keeps the solid squares and rectangles, which are then counted with contours.

In [ ]:
gray3 = cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY)
T3, bin3 = cv2.threshold(gray3, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("Otsu threshold:", T3)
show([gray3, bin3], ["Grayscale", f"Otsu threshold (T={T3:.0f}, inverted)"], cols=2, w=6, h=8)

In [ ]:
K3 = 9
m3 = morphology(bin3, K3)
show([bin3, m3["erosion"], m3["dilation"], m3["opening"], m3["closing"]],
     ["Binary", f"Erosion ({K3}x{K3})", f"Dilation ({K3}x{K3})", f"Opening ({K3}x{K3})", f"Closing ({K3}x{K3})"],
     cols=3, w=5, h=7)

In [ ]:
# Everything black (frame, ticks, text, squares...) before cleaning
n_raw, _, out_raw = count_objects(bin3, img3, min_area=1, number=False)
print("Connected black shapes before morphology:", n_raw)

# After opening: only the solid markers remain
n3, cnts3, out3 = count_objects(m3["opening"], img3, min_area=50)
areas = np.array([cv2.contourArea(c) for c in cnts3])
big = areas > 2.5 * np.median(areas)      # corner squares are much larger than side markers
print(f"Solid markers after opening: {n3}")
print(f"  large corner squares : {big.sum()}")
print(f"  small side markers   : {(~big).sum()}")

results["Picture 3: solid markers (after opening)"] = n3
results["Picture 3:   large corner squares"] = int(big.sum())
results["Picture 3:   small side markers"] = int((~big).sum())
show([out_raw, out3], [f"Before morphology: {n_raw} shapes", f"After opening: {n3} markers"],
     cols=2, w=6, h=8.5)

---
## Summary

In [ ]:
print(f"{'Measurement':50s} Count")
print("-" * 60)
for k, v in results.items():
    print(f"{k:50s} {v}")

### Notes

* **Erosion** shrinks white regions (removes specks and thin lines); **dilation** grows them (fills gaps, merges neighbours).
* **Opening** = erosion then dilation: removes small objects while keeping the size of large ones.
* **Closing** = dilation then erosion: fills small holes and gaps.
* Counts depend on `MIN_AREA`, kernel sizes and threshold values, so tune these variables and re-run.